# FTS Custom ML Backtesting Workspace

Welcome to the custom backtesting workspace! This notebook demonstrates how to run a realistic simulation of a machine learning-based trading strategy using the Financial Trading System (FTS) framework.

### Features of this Backtest:
1. **Historical Replay Event Loop:** Ticks are played back chronologically from our SQLite database.
2. **Forecasting Strategy:** Uses a pre-trained model to predict price direction from historical close prices.
3. **Execution Delay:** Simulates a realistic **execution delay** using `KBarExecuteDelay()` (meaning a signal generated at tick $T$ is submitted, and execution/fill status is checked at tick $T+k$).
4. **Price Slippage:** Simulates **price slippage** on both buy and sell orders using `FlatPriceSlip`.
5. **Visual Inspection:** Uses the `BacktestVisualizer` to display predictions overlaid with buy/sell trade markers (derived cleanly from the filled `order_logs`).
6. **Performance Summary Metrics:** Calculates and exports annualized returns, Sharpe ratio, and maximum drawdown metrics.

### 1. Import Dependencies

In [ ]:
import os
import json
import logging
import pandas as pd
import numpy as np
from datetime import datetime, timezone
from sqlalchemy import create_engine

# Core FTS components & Backtest Specification
from trading_bot.config import settings
from trading_bot.core.database import init_db, SessionLocal
from trading_bot.core.loop import HistoricalReplayLoop
from trading_bot.core.pipeline import TradingPipeline
from trading_bot.monitoring.prediction_logger import DatabasePredictionLogger
from trading_bot.core.models import BacktestPredictionLog, ModelRegistryLog, OrderLog as OrderLogModel, TradeLog as TradeLogModel, Position as PositionModel
from trading_bot.core.repository import MarketDataRepository, ModelRepository, OrderRepository, PositionRepository
from trading_bot.core.schemas import BarData, OrderSide, OrderStatus
from trading_bot.backtesting import BacktestSpec

# ML/Strategy and Risk components
from nets.output_selectors import DynamicThresholdClassifier
from nets.inference import ONNXPredictor
from nets.strategies.nets_strategy import NetsStrategy
from trading_bot.core.transforms import LogReturnTransform
from trading_bot.strategy.engine import StrategyEngine
from trading_bot.risk_management.portfolio import Portfolio
from trading_bot.risk_management.sizing.fixed_percentage import FixedPercentageSizer
from trading_bot.risk_management.manager import RiskManager

# Execution & Backtest components
from trading_bot.execution.delay import KBarExecuteDelay
from trading_bot.execution.slippage import FlatPriceSlip
from trading_bot.execution.handlers.simulated_handler import SimulatedExecutionHandler
from trading_bot.execution.engine import ExecutionEngine
from trading_bot.backtesting.readers import SQLBacktestDataReader
from trading_bot.backtesting import BacktestVisualizer, HTMLBacktestExporter

# Set logging level to INFO for detailed simulation traces
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger()

# Load canonical Backtest Specification
spec_path = '../specs/backtests/BTCUSDT/lstm_backtest.yaml'
spec = BacktestSpec.from_yaml(spec_path)
print(f"Successfully loaded canonical BacktestSpec from: {spec_path}")

### 2. Setup SQLite Database & Connect

We connect to the local SQLite database and clear any existing logs associated with our specific `run_id` to ensure a clean backtest run, without dropping the tables.

In [ ]:
db_url = 'sqlite:///../dev.db'
settings.DATABASE_URL = "sqlite+pysqlite:///../dev.db"

engine = create_engine(db_url, pool_pre_ping=True)
SessionLocal.configure(bind=engine)
db = SessionLocal()

run_id = spec.run_id

# Clear logs from previous runs of this specific backtest to ensure clean metrics
db.query(BacktestPredictionLog).filter_by(run_id=run_id).delete()
db.query(OrderLogModel).filter_by(run_id=run_id).delete()
db.query(TradeLogModel).filter_by(run_id=run_id).delete()
db.query(PositionModel).filter_by(run_id=run_id).delete()
db.commit()

print("Connected to database and cleared logs for run ID:", run_id)

### 3. Setup Strategy and Prediction Logic

We load our pre-trained model, set up feature transformation, and configure a threshold classifier output selector.

In [ ]:
with SessionLocal() as db_session:
    model_repo = ModelRepository(db_session)
    model_entry = model_repo.get_production_model(
        model_type=spec.model_type,
        market_id=spec.market.market_id,
        interval=spec.market.interval,
        horizon=spec.horizon
    )
    if model_entry is not None:
        onnx_path = model_entry.onnx_path
        print(f"Loaded production model '{model_entry.model_id}' from registry: {onnx_path}")
    else:
        model_entry = (
            db_session.query(ModelRegistryLog)
            .filter_by(model_type=spec.model_type, market_id=spec.market.market_id)
            .order_by(ModelRegistryLog.created_at.desc())
            .first()
        )
        if model_entry is not None:
            onnx_path = model_entry.onnx_path
            print(f"No production model found. Falling back to latest candidate '{model_entry.model_id}' from registry: {onnx_path}")
        else:
            onnx_path = f'../models/my_{spec.model_type}_model.onnx' if os.path.exists(f'../models/my_{spec.model_type}_model.onnx') else 'dummy.onnx'
            print(f"No model found in registry database. Falling back to default: {onnx_path}")

predictor = ONNXPredictor(onnx_path)
output_selector = DynamicThresholdClassifier(
    k=spec.classifier.classifier_k, 
    period=spec.classifier.period, 
    confidence_multiplier=spec.classifier.confidence_multiplier
)

strategy = NetsStrategy(
    predictor=predictor,
    output_selector=output_selector,
    lookback_period=predictor.model_metadata.lookback_period if predictor.model_metadata else 20,
    name_suffix=spec.model_type,
    allow_in_sample=spec.classifier.allow_in_sample
)
strategy_engine = StrategyEngine(strategies=[strategy])

### 4. Build Backtesting Pipeline with Delay and Slippage Models

Here we instantiate the components required for a realistic backtest simulation:
- **Execution Delay:** `KBarExecuteDelay(k=1)` is injected into the simulated handler.
- **Slippage:** `FlatPriceSlip(slippage_pct=0.001)` (0.1% price penalty) is injected into the simulated handler.
- **Portfolio & Sizer:** A standard portfolio initialized with $10,000, sizing positions at 10% of total equity per trade.

In [ ]:
pos_repo = PositionRepository(db)
order_repo = OrderRepository(db)

portfolio = Portfolio(
    initial_balance=spec.execution.initial_balance,
    quote_currency=spec.execution.quote_currency,
    pos_repo=pos_repo,
    order_repo=order_repo
)
portfolio._positions = {}

sizer = FixedPercentageSizer(default_percentage=spec.execution.position_size_pct)
risk_manager = RiskManager(portfolio=portfolio, sizer=sizer)

# Define delayed execution and slippage models from spec
delay_model = KBarExecuteDelay(k=spec.execution.execution_delay_k)
slippage_model = FlatPriceSlip(slippage_pct=spec.execution.slippage_pct)

execution_handler = SimulatedExecutionHandler(
    delay_model=delay_model,
    slippage_model=slippage_model,
    execution_price_source="close",
    initial_balances={spec.execution.quote_currency: spec.execution.initial_balance}
)

execution_engine = ExecutionEngine(
    execution_handler=execution_handler,
    portfolio=portfolio,
    run_id=run_id
)

pipeline = TradingPipeline(
    ingestion=None,
    strategy=strategy_engine,
    risk=risk_manager,
    execution=execution_engine,
    portfolio=portfolio
)

prediction_logger = DatabasePredictionLogger(
    db=db,
    commit=False,
    model_class=BacktestPredictionLog,
    run_id=run_id
)
pipeline.prediction_logger = prediction_logger

### 5. Run Replay Loop & Record Portfolio Equity

We initialize the data reader to stream BTC/USDT bars between June 1st, 2026, and June 21st, 2026. During the execution of the replay loop, we record the portfolio's cash, position value, and total equity at each tick to construct our equity curve.

In [ ]:
data_reader = SQLBacktestDataReader(
    session=db,
    market_id=spec.market.market_id,
    start_date=spec.dates.start_date,
    end_date=spec.dates.end_date,
    warmup_bars=spec.warmup_bars,
    lookback_limit=spec.lookback_limit
)
loop_driver = HistoricalReplayLoop(data_reader=data_reader)

print("Starting simulation loop...")
from trading_bot.backtesting.engine import BacktestEngine

# Initialize and Run Backtest Engine
backtest_engine = BacktestEngine(
    pipeline=pipeline,
    data_reader=data_reader,
    db=db,
    market_id=spec.market.market_id
)

# Run simulation and clear previous DB entries matching run_id
result = backtest_engine.run(run_id=run_id, clear_previous_run=True)

# Extract and Save Performance Summary Stats
summary = result.to_dict()
os.makedirs(spec.output_dir, exist_ok=True)
result.save_summary(os.path.join(spec.output_dir, f"backtest_summary_{run_id}.json"))

print("--- BACKTEST SUMMARY STATS ---")
print(json.dumps(summary, indent=4))

### 7. Render Interactive Dashboard

We load our interactive `BacktestVisualizer` pointing to the SQLite database and render the interactive dashboard to visually inspect cumulative returns, positions, and trades overlaying the candlestick chart.

In [ ]:
# Instantiate the visualizer pointing to the database
viz = BacktestVisualizer('sqlite:///../dev.db')

# Display the dashboard (incorporates ONNX model structure details if available)
viz.show_dashboard(onnx_model_path=onnx_path)

### 8. Export Standalone Interactive Visualization Report

Finally, we write the entire interactive visualization charts out to a standalone HTML file inside `runs/reports/` for offline review.

In [ ]:
exporter = HTMLBacktestExporter(visualizer=viz)
report_path = exporter.export(
    market_id=spec.market.market_id,
    strategy_name=strategy.name,
    run_id=run_id,
    output_path=spec.output_dir
)
print("Interactive HTML report successfully exported to:", report_path)
db.close()